<a href="https://colab.research.google.com/github/ekrombouts/GenCareAI/blob/work_in_progress/410_CarePlan_mobility_TrainFietje.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Care_Plan_Mobility: Train Fietje model

**Author:** Eva Rombouts  
**Date:** 2024-09-16  

### Description
This notebook is almost fully copied from: [Optimizing Phi-2: A Deep Dive into Fine-Tuning Small Language Models](https://medium.com/thedeephub/optimizing-phi-2-a-deep-dive-into-fine-tuning-small-language-models-9d545ac90a99), by Praveen Yerneni. Thank you!!
It trains the chat version of [Fietje](https://huggingface.co/BramVanroy/fietje-2-chat), an adapated version of microsoft/phi-2, trained on Dutch texts.

## Setup

In [ ]:
# #Install the required packages
!pip install -q bitsandbytes flash_attn datasets peft

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

import os
from google.colab import drive
from sklearn.model_selection import train_test_split
from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd
import numpy as np
import time

In [ ]:
model_name = "BramVanroy/fietje-2b-chat"
model_finetuned = "fietje_zorgplan_magweg" #Name of the model to save to HuggingFace hub
commit_message = "Trained with whole dataset, 20 epochs, truncated notes"
# random_seed = 6
# sample_size = 20

## Load model and tokenizer
Model: [fietje-2-chat](https://huggingface.co/BramVanroy/fietje-2-chat)

The model is loaded in `4-bit` which is the "Quantization" part of QLORA. The memory footprint of this is much smaller then the default.


In [ ]:
# Configuration to load model in 4-bit quantized
bnb_config = BitsAndBytesConfig(load_in_4bit=True,
                                bnb_4bit_quant_type='nf4',
                                bnb_4bit_compute_dtype='float16',
                                #bnb_4bit_compute_dtype=torch.bfloat16,
                                bnb_4bit_use_double_quant=True)


#Loading the model with compatible settings
model = AutoModelForCausalLM.from_pretrained(model_name, device_map='auto',
                                             quantization_config=bnb_config,
                                             attn_implementation='flash_attention_2',
                                             trust_remote_code=True)

# Setting up the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          add_eos_token=True,
                                          trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.truncation_side = 'left'

print(f"Memory footprint: {model.get_memory_footprint() / 1e9} GB")

## Prepare data

ToDo: Publish dataset on HF. [link text](*https://*)

In [ ]:
path_hf_sampc = "ekrombouts/Galaxy_SAMPC"
dataset = load_dataset(path_hf_sampc)

dataset

In [ ]:
train_dataset = dataset['train']
val_dataset = dataset['validation']

In [ ]:
# print(val_dataset['notes'][0])

In [ ]:
# Dimension analysis

# General function for tokenizing a column in a dataset
def tokenize_column(dataset, column_name, template=None):
    """
    Tokenize a specific column in the dataset, optionally using a template for the prompt.

    Args:
    - dataset: the dataset to process
    - column_name: the column to tokenize
    - template: a template string to wrap the column data (optional)

    Returns:
    - A list of token lengths for each entry in the specified column
    """
    token_lens = []

    for entry in dataset:
        text = entry[column_name]

        # If a template is provided, format the text using the template
        if template:
            text = template.format(text=text)

        # Tokenize the text
        tokens = tokenizer(text, return_tensors="np", truncation=False)

        # Append the length of the tokenized input
        token_lens.append(len(tokens["input_ids"][0]))

    return token_lens

# Template for the 'mobiliteit' prompt
mobiliteit_template = '''###System:
Lees de gegeven rapportages en beschrijf de mobiliteit volgens de instructies.
###Rapportages:
[rapportages hier]
###Instructies:
Beschrijf de mobiliteit van client (bv rolstoelafhankelijk, gebruik rollator, valgevaar)
###Mobiliteit:
{text}
'''

print(f"Max length for this model: {model.config.max_position_embeddings}\n")

# Tokenize the 'mobiliteit' column in the dataset with the template
mobiliteit_token_lengths = tokenize_column(train_dataset, 'mobiliteit', template=mobiliteit_template)
# mobiliteit_token_lengths = tokenize_column(train_dataset, 'mobiliteit')

# Calculate the maximum token length for 'mobiliteit'
max_mobiliteit_tokens = np.max(mobiliteit_token_lengths)
print(f"Max token length for 'mobiliteit': {max_mobiliteit_tokens}\n")

# Tokenize the 'notes' column without a template
notes_token_lengths = tokenize_column(train_dataset, 'notes')

# Calculate the statistics for the 'notes' column
notes_mean = np.mean(notes_token_lengths)
notes_min = np.min(notes_token_lengths)
notes_max = np.max(notes_token_lengths)

# Print the statistics for the 'notes' column
print(f"Mean number of tokens in 'notes': {notes_mean}")
print(f"Min number of tokens in 'notes': {notes_min}")
print(f"Max number of tokens in 'notes': {notes_max}")

In [ ]:
train_dataset

In [ ]:
def truncate_notes_to_fit_prompt(notes, max_length=1800):
    """
    Tokenizes and truncates the 'notes' text to fit within the remaining token limit.
    """
    # Tokenize the 'notes' and truncate it to the max_length
    tokens = tokenizer(notes, return_tensors="np", truncation=True, max_length=max_length)

    # Decode the truncated tokens back into text
    truncated_notes = tokenizer.decode(tokens["input_ids"][0], skip_special_tokens=True)

    return truncated_notes

# Function to compute the 'truncated_notes' column
def add_truncated_notes(example):
    notes_text = example["notes"]
    # Truncate the 'notes' to fit within the 1800 token limit
    truncated_notes = truncate_notes_to_fit_prompt(notes_text, max_length=1800)
    # Return the new field 'truncated_notes'
    return {"truncated_notes": truncated_notes}

# Add a new column 'truncated_notes' to the train_dataset
train_dataset = train_dataset.map(add_truncated_notes)

# Add a new column 'truncated_notes' to the val_dataset
val_dataset = val_dataset.map(add_truncated_notes)

In [ ]:
#Function that creates a prompt from input and output and tokenizes it
def collate_and_tokenize(examples):

    notes = examples["truncated_notes"][0]
    mobiliteit = examples["mobiliteit"][0]

    #Merging into one prompt for tokenization and training
    prompt = f'''###System:
Lees de gegeven rapportages en beschrijf de mobiliteit volgens de instructies.
###Rapportages:
{notes}
###Instructies:
Beschrijf de mobiliteit van client (bv rolstoelafhankelijk, gebruik rollator, valgevaar)
###Mobiliteit:
{mobiliteit}
'''

    #Tokenize the prompt
    encoded = tokenizer(
        prompt,
        return_tensors="np",
        padding="max_length",
        truncation=True,
        ## Very critical to keep max_length at 1024.
        ## Anything more will lead to OOM on T4
        max_length=2048,
    )

    encoded["labels"] = encoded["input_ids"]
    return encoded

In [ ]:
#We will just keep the input_ids and labels that we add in function above.
columns_to_remove = ['ct_id', 'month', 'notes', 'somatiek', 'adl', 'mobiliteit', 'continentie', 'maatschappelijk', 'psychisch', 'truncated_notes']

#tokenize the training and test datasets
tokenized_dataset_train = train_dataset.map(collate_and_tokenize,
                                            batched=True,
                                            batch_size=1,
                                            remove_columns=columns_to_remove)
tokenized_dataset_val = val_dataset.map(collate_and_tokenize,
                                          batched=True,
                                          batch_size=1,
                                          remove_columns=columns_to_remove)


In [ ]:
#Check if tokenization looks good
input_ids = tokenized_dataset_train[5]['input_ids']

decoded = tokenizer.decode(input_ids, skip_special_tokens=True)
print(decoded)

In [ ]:
len(tokenized_dataset_train[5]['input_ids'])

## Train model

### Prepare model

In [ ]:
#Accelerate training models on larger batch sizes, we can use a fully sharded data parallel model.
from accelerate import FullyShardedDataParallelPlugin, Accelerator
from torch.distributed.fsdp.fully_sharded_data_parallel import FullOptimStateDictConfig, FullStateDictConfig

fsdp_plugin = FullyShardedDataParallelPlugin(
    state_dict_config=FullStateDictConfig(offload_to_cpu=True, rank0_only=False),
    optim_state_dict_config=FullOptimStateDictConfig(offload_to_cpu=True, rank0_only=False),
)

accelerator = Accelerator(fsdp_plugin=fsdp_plugin)

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param:.2f}"
    )

In [ ]:
print_trainable_parameters(model)

#gradient checkpointing to save memory
model.gradient_checkpointing_enable()

# Freeze base model layers and cast layernorm in fp32
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
print(model)

In [ ]:
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
    'q_proj',
    'k_proj',
    'v_proj',
    'dense',
    'fc1',
    'fc2',
    ], #print(model) will show the modules to use
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

lora_model = get_peft_model(model, config)
print_trainable_parameters(lora_model)

lora_model = accelerator.prepare_model(lora_model)

### Train and save model


In [ ]:
training_args = TrainingArguments(
    output_dir='./results',  # Output directory for checkpoints and predictions
    report_to='none',
    overwrite_output_dir=True, # Overwrite the content of the output directory
    per_device_train_batch_size=4,  # Batch size for training
    per_device_eval_batch_size=4,  # Batch size for evaluation
    gradient_accumulation_steps=5, # number of steps before optimizing
    gradient_checkpointing=True,   # Enable gradient checkpointing
    gradient_checkpointing_kwargs={"use_reentrant": False},
    warmup_steps=50,  # Number of warmup steps
    #max_steps=1000,  # Total number of training steps
    num_train_epochs=20,  # Number of training epochs
    learning_rate=5e-5,  # Learning rate
    weight_decay=0.01,  # Weight decay
    optim="paged_adamw_8bit", #Keep the optimizer state and quantize it
    fp16=True, #Use mixed precision training
    #For logging and saving
    logging_dir='./logs',
    logging_strategy="steps",
    logging_steps=20,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,  # Limit the total number of checkpoints
    eval_strategy="steps",
    eval_steps=20,
    load_best_model_at_end=True, # Load the best model at the end of training
)

trainer = Trainer(
    model=lora_model,
    train_dataset=tokenized_dataset_train,
    eval_dataset=tokenized_dataset_val,
    args=training_args,
)

#Disable cache to prevent warning, reenable for inference
model.config.use_cache = False

start_time = time.time()  # Record the start time
trainer.train()  # Start training
end_time = time.time()  # Record the end time

training_time = end_time - start_time  # Calculate total training time

print(f"Training completed in {training_time} seconds.")

#Save model to hub to ensure we save our work.
lora_model.push_to_hub(model_finetuned,
                  use_auth_token=True,
                  commit_message=commit_message,
                  private=True)


#Terminate the session so we do not incur cost
from google.colab import runtime
runtime.unassign()

# Run Inference

**Note**: Ensure to stop your session and reconnect and reload the model before running the code below.

First we will run inference without the trained weights and check the output.

In [ ]:
#Setup a prompt that we can use for testing
index = 3

new_prompt = f'''###System:
Lees de gegeven rapportages en beschrijf de mobiliteit volgens de instructies.
###Rapportages:
{val_dataset["truncated_notes"][index]}
###Instructies:
Beschrijf de mobiliteit van client (bv rolstoelafhankelijk, gebruik rollator, valgevaar)
###Mobiliteit:
'''

In [ ]:
# print(new_prompt)

In [ ]:
inputs = tokenizer(new_prompt, return_tensors="pt",
                   return_attention_mask=False,
                   padding=True, truncation=True)

inputs.to('cuda')

outputs = model.generate(**inputs, repetition_penalty=1.0,
                              max_length=2048)
result = tokenizer.batch_decode(outputs, skip_special_tokens=True)

print(result[0])

In [ ]:
from peft import PeftModel, PeftConfig

#Load the model weights from hub
model_id = "ekrombouts/"+ model_finetuned
trained_model = PeftModel.from_pretrained(model, model_id)

#Run inference
outputs = trained_model.generate(**inputs, max_length=2048)
text = tokenizer.batch_decode(outputs,skip_special_tokens=True)[0]
print(text)


In [ ]:
#Setup a prompt that we can use for testing
index = 4

new_prompt = f'''###System:
Lees de gegeven rapportages en beschrijf de mobiliteit volgens de instructies.
###Rapportages:
{val_dataset["truncated_notes"][index]}
###Instructies:
Beschrijf de mobiliteit van client (bv rolstoelafhankelijk, gebruik rollator, valgevaar)
###Mobiliteit:
'''

In [ ]:
new_prompt = f'''###System:
Lees de gegeven rapportages en beschrijf de mobiliteit volgens de instructies.
###Rapportages:
- Mw heeft meegedaan met gymnastiek
- Loopt vlot en goed
- Begrijpt niet helemaal wat er gezegd wordt.
###Instructies:
Beschrijf de mobiliteit van client (bv rolstoelafhankelijk, gebruik rollator, valgevaar)
###Mobiliteit:
'''

In [ ]:
# And infer on the trained model
inputs = tokenizer(new_prompt, return_tensors="pt",
                   return_attention_mask=False,
                   padding=True, truncation=True)

inputs.to('cuda')
outputs = trained_model.generate(**inputs, max_length=2048)
text = tokenizer.batch_decode(outputs,skip_special_tokens=True)[0]

# Assuming 'text' contains the entire decoded result
split_text = text.split("###Mobiliteit:")

# Check if the split was successful and print everything after "###Mobiliteit:"
if len(split_text) > 1:
    mobility_text = split_text[1].strip()  # Strip to remove any leading/trailing spaces
    print(mobility_text)
else:
    print("###Mobiliteit: not found in text")